In [1]:
import pandas as pd
import sqlite3

In [2]:
cnx = sqlite3.connect('./data/sql-murder-mystery.db')
def sql_query(query):
    return pd.read_sql(query, cnx)

In [3]:
#Vamos a ver que tablas tenemos
sql_query("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,crime_scene_report
1,drivers_license
2,facebook_event_checkin
3,interview
4,get_fit_now_member
5,get_fit_now_check_in
6,solution
7,income
8,person


In [4]:
#Necesitamos buscar en la ciudad SQL city que asesinatos hay
query = '''
SELECT *
FROM crime_scene_report
WHERE city = 'SQL City'
  AND type = 'murder'
'''
sql_query(query)

,date,type,description,city
0,20180215,murder,REDACTED REDACTED REDACTED,SQL City
1,20180215,murder,Someone killed the guard! He took an arrow to ...,SQL City
2,20180115,murder,Security footage shows that there were 2 witne...,SQL City


In [5]:
#Ahora miramos en esos 3 resultados cual coincide con la fecha del asesinato
query = '''
SELECT *
FROM crime_scene_report
WHERE city = 'SQL City'
  AND type = 'murder'
  AND date = 20180115
'''
sql_query(query)

,date,type,description,city
0,20180115,murder,Security footage shows that there were 2 witne...,SQL City


In [6]:
#Para poder ver el desciption completo
pd.set_option('display.max_colwidth', None)

In [7]:
#Repito para ahora si ver completa la descripción
query = '''
SELECT *
FROM crime_scene_report
WHERE city = 'SQL City'
  AND type = 'murder'
  AND date = 20180115
'''
sql_query(query)

,date,type,description,city
0,20180115,murder,"Security footage shows that there were 2 witnesses. The first witness lives at the last house on ""Northwestern Dr"". The second witness, named Annabel, lives somewhere on ""Franklin Ave"".",SQL City


In [8]:
#Vamos a buscar al primer testigo
query = '''
SELECT *
FROM person
WHERE address_street_name = 'Northwestern Dr'
ORDER BY address_number DESC
LIMIT 1
'''
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


In [9]:
#Buscamos al segundo testigo:
query = '''
SELECT *
FROM person
WHERE name LIKE 'Annabel%'
  AND address_street_name = 'Franklin Ave'
'''
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


In [10]:
#Vamos a ver que declaró cada uno:
query = '''
SELECT *
FROM interview
WHERE person_id IN (14887, 16371)
'''
sql_query(query)

,person_id,transcript
0,14887,"I heard a gunshot and then saw a man run out. He had a ""Get Fit Now Gym"" bag. The membership number on the bag started with ""48Z"". Only gold members have those bags. The man got into a car with a plate that included ""H42W""."
1,16371,"I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th."


In [11]:
#Buscamos a los socios del gimnasio que encajen
#Primero miembros de Get Fit Now Gym(gold y id empiece por 48Z)
query = '''
SELECT *
FROM get_fit_now_member
WHERE id LIKE '48Z%'
  AND membership_status = 'gold'
'''
sql_query(query)

,id,person_id,name,membership_start_date,membership_status
0,48Z7A,28819,Joe Germuska,20160305,gold
1,48Z55,67318,Jeremy Bowers,20160101,gold


In [12]:
#Comprobamos quien estuvo el 9 de enero
query = '''
SELECT *
FROM get_fit_now_check_in
WHERE membership_id IN ('48Z7A', '48Z55')
  AND check_in_date = 20180109
'''
sql_query(query)

,membership_id,check_in_date,check_in_time,check_out_time
0,48Z7A,20180109,1600,1730
1,48Z55,20180109,1530,1700


In [13]:
#Ambos estuvieron el 9 de enero asi que intentamos con el coche con matrícula que contenía "H42W"
query = '''
SELECT p.name, dl.plate_number
FROM person p
JOIN drivers_license dl
ON p.license_id = dl.id
WHERE p.id IN (28819, 67318)
'''
sql_query(query)

,name,plate_number
0,Jeremy Bowers,0H42W2


In [14]:
#Ahora interrogamos al asesino para asegurar que es el al coincidir todas las pistas:
query = '''
SELECT *
FROM interview
WHERE person_id = 67318
'''
sql_query(query)

,person_id,transcript
0,67318,"I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5"" (65"") or 5'7"" (67""). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.\n"


In [15]:
#Le encargaron el asesinato asi que vamos a por la sospechosa con sus datos de persona, su licencia de conducir y asistencia a conciertos:
query = '''
SELECT p.name, COUNT(*) as veces
FROM person p
JOIN drivers_license dl ON p.license_id = dl.id
JOIN facebook_event_checkin fb ON p.id = fb.person_id
WHERE dl.gender = 'female'
  AND dl.hair_color = 'red'
  AND dl.height BETWEEN 65 AND 67
  AND dl.car_make = 'Tesla'
  AND dl.car_model = 'Model S'
  AND fb.event_name = 'SQL Symphony Concert'
  AND fb.date BETWEEN 20171201 AND 20171231
GROUP BY p.name
HAVING veces = 3
'''
sql_query(query)

,name,veces
0,Miranda Priestly,3
